# 课后练习解答（06.03_operator_analysis）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 如果 running_var 被错误设为 0 且 eps 很小，BN 折叠的主要风险是？
A. 权重被放大导致数值不稳定
B. 模型必然无法加载
C. 输出 shape 改变
D. 无风险

**解答：** A

**解析：** 分母 sqrt(0+eps) 很小时 scale 极大，权重会被异常放大。


### 问题2（单选题）

**题目：** 融合 Depthwise Conv 时忘记保持 groups 会造成？
A. 输出错误或 shape 不匹配
B. 精度更高
C. 无影响
D. 只影响速度

**解答：** A

**解析：** groups 决定通道连接方式，错误会改变算子语义，导致输出错误。


### 问题3（单选题）

**题目：** find_conv_bn_pairs 允许 Conv+BN 后还有 ReLU 的原因是？
A. 融合不改变激活层，应保留后续结构
B. ReLU 会被 BN 吸收
C. 无需保留
D. 只统计前两层

**解答：** A

**解析：** BN 折叠只替换 Conv+BN，ReLU 等激活必须保留在原位置。


### 问题4（多选题）

**题目：** eval 折叠使用 BN 的哪些参数？
A. weight
B. bias
C. running_mean
D. running_var

**解答：** ABCD

**解析：** gamma、beta、running_mean、running_var 全部进入折叠公式。


### 问题5（多选题）

**题目：** 以下哪些错误会导致融合前后输出不一致？
A. 未调用 model.eval()
B. 误用当前 batch 统计量
C. 未复制原始 conv 权重
D. 缺少 torch.no_grad()

**解答：** ABC

**解析：** no_grad 只影响 autograd 与原地写限制，不影响数值正确性。


### 问题6（判断题）

**题目：** BN 折叠是数学等价变换，融合前后输出逐 bit 完全一致。

**解答：** 错

**解析：** 浮点运算顺序改变会产生舍入误差，通常以最大误差阈值校验。


### 问题7（判断题）

**题目：** running_mean 与 running_var 在 eval 模式下不会更新。

**解答：** 对

**解析：** eval 模式只使用 running 统计量，不再用当前 batch 更新。


### 问题8（填空题）

**题目：** 折叠后新权重 W' = ____。

**解答：** gamma*W/sqrt(running_var+eps)


### 问题9（填空题）

**题目：** 折叠后新偏置 b' = ____。

**解答：** gamma*(b-running_mean)/sqrt(running_var+eps)+beta


### 问题10（简答题）

**题目：** 为什么训练模式不能直接折叠 BN？

**解答：** 训练时 BN 使用每个 batch 的动态均值方差，统计量每步变化；若提前折进权重，权重会随 batch 统计漂移，梯度路径也不再等价。


### 问题11（简答题）

**题目：** 为什么递归融合必须 setattr 写回？

**解答：** 融合会创建新的子模块对象，若不写回父模块，父模块仍引用旧子模块，深层替换不会生效。


### 问题12（代码设计题）

**题目：** 实现 find_conv_bn_pairs(model)，返回所有 Conv2d 后紧跟 BatchNorm2d 的 (父模块, conv, bn)。

**解答：** ```python
def find_conv_bn_pairs(model):
    pairs = []
    for name, module in model.named_modules():
        children = list(module.children())
        if len(children) >= 2 and isinstance(children[0], nn.Conv2d) and isinstance(children[1], nn.BatchNorm2d):
            pairs.append((module, children[0], children[1]))
    return pairs
```


### 问题13（单选题）

**题目：** nn.BatchNorm2d 的 weight/bias 参数 shape 是？
A. (C,)
B. (1,C,1,1)
C. (C,1,1)
D. (N,C)

**解答：** A

**解析：** BN 的 affine 参数按通道存储，shape 为 (C,)，计算时自动广播。


### 问题14（多选题）

**题目：** 融合后应保留的模块包括？
A. 激活层
B. BatchNorm
C. SE 模块
D. 池化层

**解答：** ACD

**解析：** BN 被吸收进 Conv，激活、SE、池化等结构继续保留。


### 问题15（简答题）

**题目：** 融合前 total=96、融合后 total=49，请解释 Conv 数量不变而 BN 全部消失。

**解答：** 每个 BN 被折叠进对应 Conv 的权重与偏置，不产生新 Conv；因此 Conv 数保持 49 不变，而 47 个 BN 全部被移除，总模块数从 96 降到 49。
